# Practical 11: A Robot That Checks Your Work

**Continuous integration with GitHub Actions, rehearsed on your own machine**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L24-L25 |
| **Course Outcome** | CO4 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 350 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. State what continuous integration is and which problem it addresses.
2. Write a GitHub Actions workflow that installs, tests and checks a project.
3. Rehearse the pipeline's own steps locally before pushing anything.
4. Observe the pipeline refusing a change, and state why that refusal is the
   point.

## What this practical is about

A test suite that is run when somebody remembers is a test suite with unknown
coverage of recent changes. **Continuous integration** removes the act of
remembering: the suite runs on every push, on a machine nobody configured, and
its result is attached to the change rather than to the person.

Three properties make it useful.

- The run happens on a **fresh machine**. Anything that works only because of
  something installed on one laptop fails there, which is precisely the class
  of fault a local run cannot detect.
- The result is a **gate**, not a notification. A later stage declares
  `needs:` on an earlier one and does not start if that one failed. L25
  measured this: with tests passing the third stage ran and exited 0, and with
  tests failing it was never attempted and the run exited 1.
- The decision is read from an **exit code**. Nothing in the pipeline
  interprets the wording of a failure.

This practical rehearses the `run:` steps locally first. That is not a
substitute for the real pipeline; it is a way to find the obvious faults in
seconds rather than in a push and a wait.

## Before you start

P05 should be complete. Tests can be written and pytest can be run.

YAML has appeared twice already. The same rule holds: spaces only.

A GitHub account is useful for the optional final step. Nothing required today
depends on one.

Run the cells from top to bottom. If notebook state becomes confusing, restart
the kernel, clear all outputs, and run again from the first cell.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['yaml', 'pytest']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['yaml', 'pytest']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: build a small project worth checking

Continuous integration requires something to test. The structure from P05 is
reused: a module, its tests, and a requirements file with pinned versions.

In [ ]:
from pathlib import Path

WORK = Path("work")
WORK.mkdir(exist_ok=True)

(WORK / "orders.py").write_text('''"Helpers for delivery orders."

TRAFFIC_LEVELS = (1, 2, 3)


def minutes_per_km(delivery_min, distance_km):
    "How many minutes each kilometre took."
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km


def is_valid_order(order):
    "True if this order is usable for training."
    return (order.get("distance_km", 0) > 0
            and order.get("prep_time_min", 0) >= 0
            and order.get("traffic_level") in TRAFFIC_LEVELS
            and order.get("rain") in (0, 1))
''', encoding="utf-8")

(WORK / "test_orders.py").write_text('''"Tests for orders.py."

import pytest

from orders import is_valid_order, minutes_per_km


def test_minutes_per_km():
    assert minutes_per_km(30, 10) == 3.0


def test_zero_distance_raises():
    with pytest.raises(ValueError):
        minutes_per_km(30, 0)


def test_good_order():
    assert is_valid_order({"distance_km": 5.0, "prep_time_min": 20,
                           "traffic_level": 2, "rain": 0})


def test_bad_traffic():
    assert not is_valid_order({"distance_km": 5.0,
                               "prep_time_min": 20,
                               "traffic_level": 9, "rain": 0})
''', encoding="utf-8")

(WORK / "requirements.txt").write_text(
    "pytest==9.1.1\nscikit-learn==1.9.0\npandas==2.3.3\n",
    encoding="utf-8")

for f in sorted(WORK.glob("*")):
    if f.is_file():
        print(" ", f.name)

## Step 2: write the workflow

The cell below holds a real GitHub Actions file. Read it as four questions:
when does it run, where does it run, what does it do, and in what order.

`uses:` draws in a prepared action written by somebody else. `checkout`
retrieves the code and `setup-python` installs an interpreter. `run:` is a
shell command of your own.

The `name:` on each step is what appears in the web interface, so each one
should state what the step establishes.

In [ ]:
workflow = """name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Get the code
        uses: actions/checkout@v4

      - name: Install Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install the libraries
        run: pip install -r requirements.txt

      - name: Run the tests
        run: pytest -q

      - name: Check the config file is valid
        run: python -c "import yaml, pathlib; yaml.safe_load(pathlib.Path('config.yaml').read_text())"
"""

flow_dir = WORK / ".github" / "workflows"
flow_dir.mkdir(parents=True, exist_ok=True)
(flow_dir / "ci.yml").write_text(workflow, encoding="utf-8")

print(workflow)

`runs-on: ubuntu-latest` deserves attention. The pipeline runs on Linux, on a
machine created moments earlier with nothing installed. That is the property
being purchased: a change that works there does not depend on anything
particular to one laptop.

## Step 3: confirm the workflow is valid YAML before pushing it

A workflow containing a YAML error frequently fails quietly. GitHub may simply
ignore the file, and the symptom is that nothing ran rather than that something
failed. Checking locally avoids that.

One property of the format is worth knowing. Under YAML 1.1 a bare `on` is read
as the boolean `True`, so after parsing, the trigger section appears under the
key `True` rather than under `"on"`. This is the same rule that turns `rain: no`
into `False` in L10, and it surprises everyone once.

In [ ]:
import yaml

with open(flow_dir / "ci.yml", encoding="utf-8") as fh:
    flow = yaml.safe_load(fh)

print("top-level keys:", list(flow))

triggers = flow.get("on", flow.get(True))
print("triggers      :", list(triggers))
print("jobs          :", list(flow["jobs"]))

steps = flow["jobs"]["test"]["steps"]
print()
print(f"{len(steps)} steps:")
for i, step in enumerate(steps, 1):
    kind = "uses" if "uses" in step else "run"
    print(f"  {i}. [{kind:4}] {step.get('name', '(unnamed)')}")

## Step 4: build a runner for the steps that can be rehearsed

A `uses:` step requires GitHub's own machinery and is therefore skipped here.
Every `run:` step is a shell command, and those can be rehearsed locally, in
order, stopping at the first failure, exactly as the real pipeline does.

One adjustment is required. On the pipeline's fresh Linux machine, `python` and
`pip` resolve to the intended interpreter. On a laptop `python` may resolve to a
different installation entirely, and on Windows it is frequently a stub from the
Microsoft Store. The runner therefore rewrites those words to point at the exact
interpreter running this notebook.

That difference is itself the lesson. This machine is not the pipeline's
machine, which is the reason the pipeline exists.

In [ ]:
import subprocess
import sys

HERE_PY = f'"{sys.executable}"'
TRANSLATIONS = [
    ("pip", f"{HERE_PY} -m pip"),
    ("pytest", f"{HERE_PY} -m pytest"),
    ("python", HERE_PY),
]


def for_this_machine(command):
    """Rewrite a CI command so it uses this notebook's Python."""
    for word, replacement in TRANSLATIONS:
        if command == word or command.startswith(word + " "):
            return replacement + command[len(word):]
    return command


def run_pipeline(steps, cwd=WORK):
    """Run the `run:` steps in order and stop at the first failure."""
    results = []
    for step in steps:
        name = step.get("name", "(unnamed)")
        if "run" not in step:
            print(f"[skip] {name}  (needs GitHub: {step['uses']})")
            results.append({"name": name, "status": "skipped"})
            continue

        command = step["run"].strip()
        print(f"[run ] {name}")
        print(f"       $ {command}")
        done = subprocess.run(for_this_machine(command), cwd=cwd,
                              shell=True, capture_output=True,
                              text=True)
        ok = done.returncode == 0
        tail = (done.stdout + done.stderr).strip().splitlines()[-4:]
        for line in tail:
            print("       |", line)
        print(f"       -> exit {done.returncode} "
              f"{'PASS' if ok else 'FAIL'}")
        results.append({"name": name,
                        "status": "pass" if ok else "fail",
                        "exit_code": done.returncode})
        if not ok:
            print()
            print("PIPELINE FAILED. Later steps do not run.")
            break
        print()
    return results


print("runner ready")

## Step 5: rehearse the pipeline

Run it. The install step draws on libraries already present, so it completes
quickly. The test step is the one that carries information.

The final step will fail, because there is no `config.yaml` in this folder.
That is deliberate. A pipeline that has never been observed to fail has not
been tested, only written.

In [ ]:
results = run_pipeline(steps)

print("=" * 55)
for r in results:
    print(f"  {r['status']:<8} {r['name']}")

Read the failure. It names the step, shows the command, and shows the error.
That is the same information the red cross in the web interface presents, which
means a fault of this kind can be diagnosed here in seconds rather than through
a push and a wait.

## Step 6: repair the pipeline

The pipeline required a `config.yaml` that does not exist. Two honest remedies
are available: create the file, or stop asking for it. The file is created here,
because a real project would have one and because removing a check to make a
pipeline green is the wrong instinct to establish.

In [ ]:
(WORK / "config.yaml").write_text(
    "split:\n  test_size: 0.2\n  seed: 42\n", encoding="utf-8"
)

results = run_pipeline(steps)

print("=" * 55)
passed = sum(1 for r in results if r["status"] == "pass")
failed = sum(1 for r in results if r["status"] == "fail")
print(f"  {passed} passed, {failed} failed")
print("  PIPELINE GREEN" if failed == 0 else "  PIPELINE RED")

## Step 7: break a test and observe the pipeline catching it

This cell is the value of continuous integration in one place. A fault is
introduced of the kind a tired person produces, a wrong operator, and the
pipeline locates it.

In [ ]:
original = (WORK / "orders.py").read_text(encoding="utf-8")
(WORK / "orders.py").write_text(
    original.replace("return delivery_min / distance_km",
                     "return delivery_min * distance_km"),
    encoding="utf-8",
)

print("A bug has been introduced. Running the pipeline...")
print()
broken_results = run_pipeline(steps)

(WORK / "orders.py").write_text(original, encoding="utf-8")
print()
print("bug removed, orders.py restored")

The pipeline failed at the test step and the steps after it did not run. On
GitHub this is the point at which a pull request is blocked and its author is
notified. Nobody had to notice, which is the whole intent: the check does not
depend on attention.

## Step 8: reduce the run time with a cache

Installing libraries from nothing on every push is slow. GitHub can retain
pip's download cache between runs, and `setup-python` enables it with one
additional line.

Speed is not vanity here. A pipeline taking fifteen minutes is a pipeline people
learn to work around, and a gate that is routinely bypassed is not a gate.

In [ ]:
cached = yaml.safe_load(
    (flow_dir / "ci.yml").read_text(encoding="utf-8")
)
for step in cached["jobs"]["test"]["steps"]:
    if step.get("uses", "").startswith("actions/setup-python"):
        step["with"]["cache"] = "pip"

(flow_dir / "ci-cached.yml").write_text(
    yaml.safe_dump(cached, sort_keys=False), encoding="utf-8"
)

print(yaml.safe_dump(cached["jobs"]["test"]["steps"][1],
                     sort_keys=False))

## Step 9: what happens on GitHub

This need not be carried out today, but its shape should be known.

In [ ]:
print("To make this real:")
print()
print("  1. Put your project in a GitHub repository.")
print("  2. Commit the file at .github/workflows/ci.yml")
print("     -- the path must be exactly that.")
print("  3. git push")
print("  4. Open the repository in a browser and click the")
print("     'Actions' tab at the top.")
print()
print("What you will see first: a list of workflow runs, newest")
print("at the top, each with a yellow dot (running), a green tick")
print("(passed) or a red cross (failed). Click a run, then a job,")
print("to expand each step and read its output.")
print()
print("In Settings -> Branches you can then add a rule that")
print("REQUIRES this check to pass before anyone may merge into")
print("main. That is the moment CI stops being advice and starts")
print("being a rule.")

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: trigger it on the right events

Build a dictionary `T1_triggers` describing a workflow that runs

- on every push to the branches `main` and `develop`,
- on every pull request targeting `main`.

The structure must be the one GitHub expects, namely
`{"push": {"branches": [...]}, "pull_request": {"branches": [...]}}`.

> **Hint.** This is a plain Python dictionary and no YAML is required. Follow the
> `on:` block printed in Step 2 and add one further branch under `push`.

In [ ]:
# TODO: push on main and develop, pull_request on main
T1_triggers = None

print(T1_triggers)

## Task T2: add a step that would have caught a real fault

Add one further step to the workflow, named exactly
`Check the model can be imported`, running the command

```
python -c "import orders; print(orders.minutes_per_km(30, 10))"
```

Append it to the `steps` list, write the updated workflow back to
`work/.github/workflows/ci.yml`, and store the new steps list in `T2_steps`.

> **Hint.** A step is a dictionary with `name` and `run` keys. Use
> `steps.append({...})`, then rebuild the file with
> `yaml.safe_dump(flow, sort_keys=False)`. Set `T2_steps = steps`.

In [ ]:
new_step = {
    "name": "Check the model can be imported",
    # TODO: the command to run
    "run": None,
}

# TODO: append it to steps, save the workflow file, and set T2_steps
T2_steps = None

print(T2_steps)

## Task T3: rehearse it until it passes

Run the six step pipeline with `run_pipeline` and store the result list in
`T3_results`.

Every step that actually runs must pass. If one fails, read the output, correct
the cause, and run again, exactly as would be done before pushing.

> **Hint.** `T3_results = run_pipeline(T2_steps)`. If the import check fails
> with `ModuleNotFoundError`, confirm that `orders.py` is intact; Step 7
> modified it and restored it.

In [ ]:
# TODO: run the pipeline and keep the results
T3_results = None

print(T3_results)

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | both trigger kinds are present', lambda: set(T1_triggers) == {'push', 'pull_request'})
_check('T1 | push covers main and develop', lambda: sorted(T1_triggers['push']['branches']) == ['develop', 'main'])
_check('T1 | pull_request targets main only', lambda: T1_triggers['pull_request']['branches'] == ['main'])
_check('T2 | the workflow now has six steps', lambda: len(T2_steps) == 6)
_check('T2 | the last one is the import check, with a run command', lambda: T2_steps[-1]['name'] == 'Check the model can be imported' and 'run' in T2_steps[-1])
_check('T2 | its command really imports orders and calls the function', lambda: 'import orders' in T2_steps[-1]['run'] and 'minutes_per_km' in T2_steps[-1]['run'])
_check('T2 | the change was saved to the workflow file', lambda: 'Check the model can be imported' in (flow_dir / 'ci.yml').read_text(encoding='utf-8'))
_check('T3 | the pipeline ran all six steps', lambda: len(T3_results) == 6)
_check('T3 | nothing failed', lambda: not any(r['status'] == 'fail' for r in T3_results))
_check('T3 | the tests and the import check both actually ran', lambda: sum(1 for r in T3_results if r['status'] == 'pass') == 4)

print("==================================================================")
print("SELF-CHECK   Practical 11: A Robot That Checks Your Work")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The file `work/.github/workflows/ci.yml`.
3. In a markdown cell, two sentences: what the pipeline did when the test was
   broken, and why the steps after it did not run.

Name the file `P11_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, pipeline observed passing then failing | 3 |
| Task T1, correct triggers on the workflow | 2 |
| Task T2, a useful step added to the pipeline | 3 |
| Task T3, the pipeline rehearsed and passing | 2 |
| **Total** | **10** |

## Read more

- GitHub Docs, understanding GitHub Actions: <https://docs.github.com/en/actions/learn-github-actions/understanding-github-actions>
- GitHub Docs, workflow syntax: <https://docs.github.com/en/actions/using-workflows/workflow-syntax-for-github-actions>
- GitHub Docs, caching dependencies: <https://docs.github.com/en/actions/using-workflows/caching-dependencies-to-speed-up-workflows>
- actionlint, a workflow linter: <https://github.com/rhysd/actionlint>